# 20 GNEWS búsqueda exhaustiva


In [2]:
import pandas as pd

from gnews import GNews

### Dataframe con artículos buscados por GNews (sin el texto)

Palabras de la búsqueda en AppsScript:

/***********************  
   RSS búsquedas seguras  
 ***********************/  
const SEARCH_QUERIES = [  
  'detenido CABA',  
  'detenida CABA',  
  'detuvieron CABA',  
  'demorado CABA',  
  'represion policial CABA',  
  'represión policial CABA',  
  'detenido "Capital Federal"',  
  'represion policial "Capital Federal"',  
  'detenido "Ciudad de Buenos Aires"',  
  'represion policial "Ciudad de Buenos Aires"',  
  'violencia policial "Ciudad de Buenos Aires"',  
  'gatillo facil "Ciudad de Buenos Aires"',  
  'abuso policial"Ciudad de Buenos Aires"',  
  'violencia institucional "Ciudad de Buenos Aires"',  
];  

In [ ]:
# Sitios donde realizar búsquedas por sitio
sites = [
    "www.agenciapacourondo.com.ar",
    "www.c5n.com",
    "www.eldestapeweb.com",
    "www.clarin.com",
    "www.infobae.com",
    "www.laizquierdadiario.com",
    "www.lanacion.com.ar",
    "www.pagina12.com.ar",
    "www.perfil.com",
    "www.tiempoar.com.ar",
    "tn.com.ar",
]

# Condition to append to the query with the list of sites, with the goal of reducing the number of queries.
# query_sites = " OR ".join([f"site:{site}" for site in sites])
# Important: The Google News tab can sometimes cut off or fail to pull comprehensive results if the string gets excessively long. For optimal and reliable results on Google News, it is best to keep your list to 10–15 sites per search

exclude = [
    "www.facebook.com",
    "www.instagram.com",
    "www.laizquierdadiario.cl"
]

keywords = [
    # 'detenido',  
    # 'detenida',  
    # 'detuvieron',  
    # 'demorado',
    # 'demorada',
    'represión',  
    # 'represion',  
    'policial',
    'violencia policial',
    'violencia institucional',
    # 'policía',
    'abuso policial',
    'abuso fuerza de seguridad',
    'policía de la ciudad',
    'gendarmería',
    'prefectura',
    'policía federal',
    'aeroportuaria',
    'policía de tránsito',
    'fuerza de seguridad',
    'seguridad privada',
    # 'gatillo facil',  
    'gatillo fácil',
    # 'víctima',
    # 'disparo',
    # 'herido',
    # 'herida',
    # 'asesinato',
    # 'asesinada',
    # 'asesinado',
    # 'ejecutado',
    # 'ejecutada',
    # 'torturado',
    # 'torturada',
]

In [ ]:
start = (2026, 9, 4)
end = (2026, 9, 5)

# Lista de sitios web a buscar vacía para disminuir el tiempo de búsqueda.
# sites = []

# Lista reducida para disminuir el tiempo de búsqueda.
# keywords = [
#     'violencia policial',
#     'violencia institucional',
#     'abuso policial',
#     'policía de la ciudad',
#     'gendarmería',
#     'prefectura',
#     'policía federal',
#     'aeroportuaria',
#     'policía de tránsito',
#     'fuerza de seguridad',
#     'gatillo fácil',
# ]

# Número de búsquedas.
Ntotal = (len(sites) + 1) * len(keywords)
n = 1
all_articles = []
detected_keyword = []

for k in keywords:
    gn = GNews(language="es-419", country="AR")
    gn.start_date = start
    gn.end_date = end

    # Primero una búsqueda por sitio.
    for site in sites:
        print(f'\r Búsqueda {n}/{Ntotal}', end='', flush=True)
        n += 1

        articles = gn.get_news(f"{k} site:{site}")
        # articles = gn.get_news(f"{k} ({query_sites})")
        if len(articles) == 100:
            print(f"Warning: Reached 100 articles for keyword '{k}' on site '{site}'")

        detected_keyword.extend([k] * len(articles))
        # Handle pygooglenews dictionary output ('entries') or raw list
        entries = articles["entries"] if isinstance(articles, dict) else articles
        all_articles.extend(entries)

    # Ahora una búsqueda general que no esté limitada a un sitio específico.
    print(f'\r Búsqueda {n}/{Ntotal}', end='', flush=True)
    n += 1
    gn.exclude_websites = sites + exclude

    articles = gn.get_news(k)
    if len(articles) == 100:
        print(f"Warning: Reached 100 articles for keyword '{k}' in general search")

    detected_keyword.extend([k] * len(articles))
    entries = articles["entries"] if isinstance(articles, dict) else articles
    all_articles.extend(entries)

# Convert the combined list into a single DataFrame
df = pd.DataFrame(all_articles)
publisher_cols = pd.json_normalize(df['publisher']).set_index(df.index)
df = pd.concat([df.drop(columns=['publisher']), publisher_cols], axis=1)
df['keyword'] = detected_keyword
df

In [ ]:
df_unique = df.value_counts(subset=['title']).reset_index(name='count')
df_unique

In [ ]:
df.loc[df['title'] == df_unique.iloc[77]['title']]

In [ ]:
df_unique = df.value_counts(subset=['title']).reset_index(name='count')
df_unique

In [ ]:
print(df.iloc[127]['url'])


## Observaciones

- Cuando se busca 'gatillo fácil', siempre aparecen artículos de https://www.laizquierdadiario.cl aunque en los artículos no aparezca 'gatillo' ni 'fácil'.
- Incluir las palabras con y sin tilde dan resultados diferentes, por ejemplo: represión y represion. Aunque la palabra esté bien escrita en el artículo (un artículo que contiene la palabra represión), algunos artículos no aparecen en la búsqueda por 'represión' y sí aparecen cuando la búsqueda es 'represion'. Sin embargo, hacerlo con todas las palabras que tienen tilde agrega muchas búsquedas, por lo cual no se hace con todas. Es cuestión de experimentar si conviene agregarla o no.
- https://www.laizquierdadiario.cl muestra artículos de Argentina.
- revista crisis, instagram, facebook: ¿excluir?
- country='AR' , Google ignores this